<a href="https://colab.research.google.com/github/mosesdorri-gif/Project-1/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

firts things first, we have to upload pandas(the python library for data frames and manipulation), and matplotlib(to plot our visualizations)

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


Now we have to assign a variable to our dataset, this will be oc(short for crime occurances)

In [5]:
oc = pd.read_csv('/content/drive/MyDrive/Datasets/Occurrence_-4324496069323609111.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Datasets/Occurrence_-4324496069323609111.csv'

and here I remove the null values, and replace them with a much cleaner no

In [6]:
oc.fillna('no',inplace = True)

NameError: name 'oc' is not defined

In [ ]:
oc

now we want to set a filter, we are only going to be looking at the crime reports in Vaughan

In [ ]:
filt = oc['Municipality'] == 'Vaughan'

let's take a look at the data from Vaughan

In [ ]:
df = oc.loc[filt]
df

now that we have filtered all the reports that are in vaughan we can now delve down deeper into the dataset, and look specifically into the occurence types, we can count that and look at that in a series

In [7]:
o_type = df['Occurrence Type'].value_counts()
o_type

NameError: name 'df' is not defined

This is all of the types of crime reports that have been reported

the data has already been sorted, but we can gather some more insights

In [ ]:
o_type.describe()

25 diffrent types of reports, mininum for a type was 5 occurrences, maximum is 5275 reports for an occurence, and the mean amount of reports for an occurence is 834.28 while the median is 361. clearly the outliers are skewing the mean significantly. Is there more data we can gather? YES there is

In [ ]:
lo_type = df.groupby(df['Location Code'])
lo_data = lo_type['Location Code'].value_counts()
lo_data.sort_values(ascending=False,inplace = True)
lo_data.drop('no', inplace = True)
lo_data

here is the location code for all of the occurences, as we can see The most of them happen at businesses, and the least happen outdoors. we can find the percentages of each as well

In [ ]:

total_num = lo_data.sum()
lo_p = lo_data / total_num
lo_ = 'percentage(in decimal)'
lop = pd.DataFrame({'count':lo_data, 'total_percentage':lo_p})
lop

here is the data represented by its count, and by its coresponding percentage, ~ 45% of the occurences occur in busniness, ~32% in residences and ~23% of occurences happen outdoors



we can make it even more detailed and make a bar graph

In [ ]:
catagory = lop.index
values = lop['total_percentage']
plt.pie(values, labels = catagory, autopct = '%1.1f%%')
plt.title('percentage of occurences by location')

We can also make a graph representing the percantage of a specific report type, given that a report has occured

In [ ]:
cat = o_type.index
op = o_type / o_type.sum()
val = op
plt.barh(cat, val ,edgecolor = 'black')
plt.title('percentage of occurences by type')
plt.xlabel('percentage(in decimals)')
plt.ylabel(' occurrence type')

horizontal bar graph was chosen due to there being a lot of occurence types but here is the data.

Now that we have gotten some basic analytics done, lets answer an interesting question. What will the thefts under 5000$ look like in the next coming years

for starters we can map this out on a time-series plot, but that'll require some data cleaning and filtering from our dataset.

In [ ]:
lg = pd.to_datetime(df['Occurrence Date'],format = '%Y-%m-%d')

In [8]:
df.loc['Year_Month'] = lg.dt.to_period('M')

NameError: name 'lg' is not defined

In [ ]:
df.groupby('Year_Month')

In [ ]:
falt = df['Occurrence Type'] == 'Theft Under $5000'

In [ ]:
gf = df.loc[falt]

In [ ]:
gr1 = gf['Occurrence Type'].value_counts()

In [ ]:
gf.set_index(np.arange(len(gf)),inplace = True)
gf.groupby(gf['Year_Month'])
gf



In [ ]:
af = pd.pivot_table(
    df,
    values='Occurrence Type',
    index='Year_Month',
    columns= falt,
    aggfunc='count'
)
af

In [9]:
gf = af.drop(False, axis = 1)
gf['Num of Theft below 5000$'] = gf[True]
gf.drop(True, axis = 1, inplace = True)
gf.drop('2026-07', axis=0, inplace=True)

NameError: name 'af' is not defined

In [ ]:
fig = plt.figure(figsize=(17,6))
sns.scatterplot(x = gf.index.astype(str), y = 'Num of Theft below 5000$', data = gf,)

looking at the graph we can also apply the mean and median,this will give us some insight

In [ ]:
gf.agg(['mean','median'])

we can go one step further and ask ourselves, 'is there a relationship between the times of the crime, do more occurences happen during the night. we obviously would assume so, but how true could this be? well, lets first plot it out and then we can see if there is a relationship.

for a more accurate overview we'll use all of york region since it contains more data. We can also start by making a simple count plot that tracks the count of occurences given the time.

In [ ]:
oc['Time'] = pd.to_datetime(df['Time'], format='%H:%M')
oc['Time'] = oc['Time'].dt.hour

In [ ]:
fig, ax = plt.subplots(figsize=(17,6))
yt =sns.countplot(x = 'Time', data = oc, color = 'red', ax=ax,)


now that we have the data in front of us, some basic EDA tells us most occured during midnight and the least occured during 6am.